**Necessary imports**

In [1]:
import subprocess
import os
import pandas as pd

%load_ext autoreload
%autoreload 2

**Setting parameters**

In [2]:
simple_render_raw_data_dir = "/dlabdata1/tsoares/wikidumps/"
processed_data_dir = "/dlabdata1/boulenge/wikidumps_data/report_examples_3"
server_out_dir = "/var/www/html/wikinsert/report_examples_3"
processes = 48
max_samples_per_lang = 15
langs = ['fr','es', 'simple', 'de'] 

#For all possible languages
#sorted(set([filename.split("wiki")[0] for filename in os.listdir(simple_render_raw_data_dir) if "wiki" in filename]))

side_context_sizes = [5]  #List of context size to try
second_month = "20231001"
third_month = "20231101"

langs_to_remove = ["war", "ky", "nds-nl"] #These languages do not work

for lang in langs_to_remove:
    if lang in langs:
        langs.remove(lang)



**Defining data paths**

In [3]:
# Define directories
rev_files_dir = os.path.join(processed_data_dir, "revisions_files")
parsed_htmls_dir = os.path.join(processed_data_dir, "parsed_htmls")
contexts_splitted_data_dir = os.path.join(processed_data_dir, "contexts_splitted_data")
ready_to_score_data_dir = os.path.join(processed_data_dir, "ready_to_score_data")
scored_data_dir = os.path.join(processed_data_dir, "scored_data")
crawling_output_dir = os.path.join(server_out_dir, "crawled_pages")

# Create necessary directories if they don't exist
os.makedirs(rev_files_dir, exist_ok=True)
os.makedirs(parsed_htmls_dir, exist_ok=True)
os.makedirs(contexts_splitted_data_dir, exist_ok=True)
os.makedirs(ready_to_score_data_dir, exist_ok=True)
os.makedirs(scored_data_dir, exist_ok=True)
os.makedirs(crawling_output_dir, exist_ok=True)
os.makedirs(server_out_dir, exist_ok=True)

**Prepare files to be crawled and parsed**

It generates the diffenret pages ids we will crawl, using data from the previous study

In [7]:
for lang in langs:
        print(f"Processing language: {lang}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
            
        subprocess.run([
            "python", "generate_revisions_file.py",
            "--versions_file", f"{second_month_path}/eval/link_versions",
            "--html_dir", f"{second_month_path}/eval/pages",
            "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
            "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",
            "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",
            "--max_samples", str(max_samples_per_lang),
            "--save_path", save_path
        ])


Processing language: fr


Checking revisions: 100%|██████████| 26847/26847 [01:21<00:00, 330.66it/s]


Processing language: es


Checking revisions: 100%|██████████| 29970/29970 [01:14<00:00, 401.24it/s]


Processing language: simple


Checking revisions: 100%|██████████| 1841/1841 [00:27<00:00, 68.15it/s] 


Processing language: de


Checking revisions: 100%|██████████| 49899/49899 [01:59<00:00, 418.36it/s]


**Crawling first versions only**

There is no need to re-crawl the second version with all its files as the index is sufficent for these.

In [4]:
for lang in langs:
    print(f"Processing language: {lang}")
    subprocess.run([
        "python", "crawl.py",
        "--revisions_path", f"{rev_files_dir}/revisions_{lang}.parquet",
        "--output_dir", f"{crawling_output_dir}/{lang}",
        "--n_processes", str(processes),
        "--lang", lang
    ])

Processing language: fr
Total number of pages to crawl: 15
Running crawling on 48 processes
Crawling finished, post processing...


Moving files: 100%|██████████| 15/15 [00:00<00:00, 2760.98it/s]


Processing language: es
Total number of pages to crawl: 15
Running crawling on 48 processes
Crawling finished, post processing...


Moving files: 100%|██████████| 15/15 [00:00<00:00, 2118.26it/s]


Processing language: simple
Total number of pages to crawl: 15
Running crawling on 48 processes
Crawling finished, post processing...


Moving files: 100%|██████████| 15/15 [00:00<00:00, 5017.11it/s]


Processing language: de
Total number of pages to crawl: 15
Running crawling on 48 processes
Crawling finished, post processing...


Moving files: 100%|██████████| 15/15 [00:00<00:00, 4023.18it/s]


**Parsing of the htmls and sentence tokenizing**

In [46]:
for lang in langs:
    print(f"Processing language: {lang}")
    second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
    first_version_specific_html_dir = f"{crawling_output_dir}/{lang}"
    save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
    output_dir = f"{parsed_htmls_dir}/{lang}_v2"
    first_version_specific_output_dir = f"{parsed_htmls_dir}/{lang}_v1"
        
    subprocess.run([
        "python", "parsing.py",
        "--revisions_path", save_path,        
        "--html_dir", f"{second_month_path}/eval/pages",
        "--first_versions_specific_html_dir", first_version_specific_html_dir,
        "--output_dir", output_dir,
        "--first_versions_specific_output_dir", first_version_specific_output_dir,
        "--lang", lang,
        "--n_processes", "10"#str(processes)
    ])

    parsed_htmls_paths = os.listdir(first_version_specific_output_dir)
    sentences_path = list(filter(lambda x: x.endswith("sentences.jsonl"), parsed_htmls_paths))[0]
    links_path = list(filter(lambda x: x.endswith("links.jsonl"), parsed_htmls_paths))[0]
    sentences_df = pd.read_json(f"{first_version_specific_output_dir}/{sentences_path}")
    links_df = pd.read_json(f"{first_version_specific_output_dir}/{links_path}")

    display(sentences_df.head(5))
    display(links_df.head(5))   

Processing language: fr
First version html dir: /var/www/html/wikinsert/report_examples_3/crawled_pages/fr
Running on 10 processes
Parallel multiprocess
Writing sentences and links v1
{"sentence":{"0":"\"<<O0>>\\n<<C0>><<O1>>Cet article est une <<C1>><<O2>>\u00c3\u00a9bauche<<C2>><<O3>> concernant une <<C3>><<O4>>cr\u00c3\u00a9ature l\u00c3\u00a9gendaire<<C4>><<O5>>.<<C5>><<O6>>\\n<<C6>><<O7>>Vous pouvez partager vos connaissances en l\u00e2\u0080\u0099am\u00c3\u00a9liorant (<<C7>><<O8>>comment&nbsp;?<<C8>><<O9>>) selon les recommandations des <<C9>><<O10>>projets correspondants<<C10>><<O11>>.\\n<<C11>><<O12>>\\n<<C12>><<O13>>\\n<<C13>><<O14>>Plusieurs cr\u00c3\u00a9atures l\u00c3\u00a9gendaires, illustration du <<C14>><<O15>>Bilderbuch f\u00c3\u00bcr Kinder<<C15>><<O16>> (livre d'images pour enfants) de <<C16>><<O17>>Friedrich Justin Bertuch<<C17>><<O18>> (1806).<<C18>><<O19>>\\n<<C19>><<O20>>Une <<C20>><<O21>>cr\u00c3\u00a9ature l\u00c3\u00a9gendaire<<C21>><<O22>> est, dans le domain

/scratch/boulenge/wikinsert/js_parsing/html_processors.js:245
        firstControl = sentence.match(/<<[O|C]\d+>>/); //first occurence of <<0 textNodeId>> or <<C textNodeId>>
                                ^

TypeError: sentence.match is not a function
    at getSentenceStore (/scratch/boulenge/wikinsert/js_parsing/html_processors.js:245:33)
    at write_sentences_store (/scratch/boulenge/wikinsert/js_parsing/html_processors.js:491:27)
    at processFiles (/scratch/boulenge/wikinsert/js_parsing/parse.js:75:15)
    at processTicksAndRejections (node:internal/process/task_queues:96:5)


{"sentence":{"0":"\"<<O0>>\\n<<C0>><<O1>>\\n<<C1>><<O2>> <<C2>><<O3>> <<C3>><<O4>>\\n<<C4>><<O5>>Cet article est une <<C5>><<O6>>\u00c3\u00a9bauche<<C6>><<O7>> concernant un <<C7>><<O8>>film<<C8>><<O9>> fran\u00c3\u00a7ais et <<C9>><<O10>>Marseille<<C10>><<O11>>.<<C11>><<O12>>\\n<<C12>><<O13>>Vous pouvez partager vos connaissances en l\u00e2\u0080\u0099am\u00c3\u00a9liorant (<<C13>><<O14>>comment&nbsp;?<<C14>><<O15>>) selon les <<C15>><<O16>>conventions filmographiques<<C16>><<O17>>.\\n<<C17>><<O18>>\\n<<C18>><<O19>>\\n<<C19>><<O20>>\\n<<C20>><<O21>>\\n<<C21>><<O22>>L'argent fait le bonheur<<C22>><<O23>>\\n<<C23>><<O24>>\\n<<C24>><<O25>>\\n\\n<<C25>><<O26>>\\n<<C26>><<O27>>\\n<<C27>><<O28>>\\n\\n<<C28>><<O29>>\\n<<C29>><<O30>>\\n<<C30>><<O31>>Pour plus de d\u00c3\u00a9tails, voir <<C31>><<O32>> et <<C32>><<O33>>\\n<<C33>><<O34>>modifier<<C34>><<O35>> <<C35>><<O36>>\\n<<C36>><<O37>>L'argent fait le bonheur<<C37>><<O38>> est un <<C38>><<O39>>film fran\u00c3\u00a7ais<<C39>><<O40>> r\u00c3

/scratch/boulenge/wikinsert/js_parsing/html_processors.js:245
        firstControl = sentence.match(/<<[O|C]\d+>>/); //first occurence of <<0 textNodeId>> or <<C textNodeId>>
                                ^

TypeError: sentence.match is not a function
    at getSentenceStore (/scratch/boulenge/wikinsert/js_parsing/html_processors.js:245:33)
    at write_sentences_store (/scratch/boulenge/wikinsert/js_parsing/html_processors.js:491:27)
    at processFiles (/scratch/boulenge/wikinsert/js_parsing/parse.js:75:15)


KeyboardInterrupt: 

...

/scratch/boulenge/wikinsert/js_parsing/html_processors.js:245
        firstControl = sentence.match(/<<[O|C]\d+>>/); //first occurence of <<0 textNodeId>> or <<C textNodeId>>
                                ^

TypeError: sentence.match is not a function
    at getSentenceStore (/scratch/boulenge/wikinsert/js_parsing/html_processors.js:245:33)
    at write_sentences_store (/scratch/boulenge/wikinsert/js_parsing/html_processors.js:491:27)
    at processFiles (/scratch/boulenge/wikinsert/js_parsing/parse.js:75:15)
Process ForkPoolWorker-1:
Traceback (most recent call last):
  File "/home/boulenge/.conda/envs/wiki_dump/lib/python3.9/multiprocessing/pool.py", line 131, in worker
    put((job, i, result))
  File "/home/boulenge/.conda/envs/wiki_dump/lib/python3.9/multiprocessing/queues.py", line 377, in put
    self._writer.send_bytes(obj)
  File "/home/boulenge/.conda/envs/wiki_dump/lib/python3.9/multiprocessing/connection.py", line 200, in send_bytes
    self._send_bytes(m[offset:offset + 

{"sentence":{"0":"\"<<O0>> <<C0>><<O1>> <<C1>><<O2>>\\n<<C2>><<O3>>Cet article est une <<C3>><<O4>>\u00c3\u00a9bauche<<C4>><<O5>> concernant la <<C5>><<O6>>Perse antique<<C6>><<O7>> et un <<C7>><<O8>>musicien<<C8>><<O9>> <<C9>><<O10>>iranien<<C10>><<O11>>.<<C11>><<O12>>\\n<<C12>><<O13>>Vous pouvez partager vos connaissances en l\u00e2\u0080\u0099am\u00c3\u00a9liorant (<<C13>><<O14>>comment&nbsp;?<<C14>><<O15>>) selon les recommandations des <<C15>><<O16>>projets correspondants<<C16>><<O17>>.\\n<<C17>><<O18>>\\n<<C18>><<O19>>\\n<<C19>><<O20>>Barbad<<C20>><<O21>>modifier<<C21>><<O22>> - <<C22>><<O23>>modifier le code<<C23>><<O24>> - <<C24>><<O25>>modifier Wikidata<<C25>><<O26>>\\n<<C26>><<O27>>Barbad<<C27>><<O28>> ou <<C28>><<O29>>Barbad le Grand<<C29>><<O30>> (en <<C30>><<O31>>Persan<<C31>><<O32>>: \u00d8\u00a8\u00d8\u00a7\u00d8\u00b1\u00d8\u00a8\u00d8\u00af), aussi transcrit Barbod, \u00c3\u00a9tait un musicien de la cour de l'Empire <<C32>><<O33>>Sassanide<<C33>><<O34>> en <<C34>><<O3

/scratch/boulenge/wikinsert/js_parsing/html_processors.js:245
        firstControl = sentence.match(/<<[O|C]\d+>>/); //first occurence of <<0 textNodeId>> or <<C textNodeId>>
                                ^

TypeError: sentence.match is not a function
    at getSentenceStore (/scratch/boulenge/wikinsert/js_parsing/html_processors.js:245:33)
    at write_sentences_store (/scratch/boulenge/wikinsert/js_parsing/html_processors.js:491:27)
    at processFiles (/scratch/boulenge/wikinsert/js_parsing/parse.js:75:15)
Process ForkPoolWorker-5:
Traceback (most recent call last):
  File "/home/boulenge/.conda/envs/wiki_dump/lib/python3.9/multiprocessing/pool.py", line 131, in worker
    put((job, i, result))
  File "/home/boulenge/.conda/envs/wiki_dump/lib/python3.9/multiprocessing/queues.py", line 377, in put
    self._writer.send_bytes(obj)
  File "/home/boulenge/.conda/envs/wiki_dump/lib/python3.9/multiprocessing/connection.py", line 200, in send_bytes
    self._send_bytes(m[offset:offset + 

{"sentence":{"0":"\"<<O0>>\\n<<C0>><<O1>>Pour les articles homonymes, voir <<C1>><<O2>>Bi\u00c3\u00a8vre<<C2>><<O3>>.\\n<<C3>><<O4>>\\n<<C4>><<O5>>\\n<<C5>><<O6>>\\n<<C6>><<O7>>\\n<<C7>><<O8>>\\n<<C8>><<O9>>La <<C9>><<O10>>Bi\u00c3\u00a8vre<<C10>><<O11>> est une <<C11>><<O12>>rivi\u00c3\u00a8re<<C12>><<O13>> qui prend source \u00c3\u00a0 <<C13>><<O14>>Guyancourt<<C14>><<O15>> et qui se jette dans le collecteur principal des <<C15>><<O16>>\u00c3\u00a9gouts de Paris<<C16>><<O17>>.\\n<<C17>><<O18>>La Bi\u00c3\u00a8vre se jetait autrefois dans la <<C18>><<O19>>Seine<<C19>><<O20>> \u00c3\u00a0 <<C20>><<O21>>Paris<<C21>><<O22>> (au niveau de la <<C22>><<O23>>gare d'Austerlitz<<C23>><<O24>>) apr\u00c3\u00a8s un parcours de 35&nbsp;<<C24>><<O25>>km<<C25>><<O26>>[<<C26>><<O27>>1<<C27>><<O28>>]<<C28>><<O29>> dans les d\u00c3\u00a9partements des <<C29>><<O30>>Yvelines<<C30>><<O31>>, de l'<<C31>><<O32>>Essonne<<C32>><<O33>>, des <<C33>><<O34>>Hauts-de-Seine<<C34>><<O35>>, du <<C35>><<O36>>Val-de-M

/scratch/boulenge/wikinsert/js_parsing/html_processors.js:245
        firstControl = sentence.match(/<<[O|C]\d+>>/); //first occurence of <<0 textNodeId>> or <<C textNodeId>>
                                ^

TypeError: sentence.match is not a function
    at getSentenceStore (/scratch/boulenge/wikinsert/js_parsing/html_processors.js:245:33)
    at write_sentences_store (/scratch/boulenge/wikinsert/js_parsing/html_processors.js:491:27)
    at processFiles (/scratch/boulenge/wikinsert/js_parsing/parse.js:75:15)
Process ForkPoolWorker-6:
Traceback (most recent call last):
  File "/home/boulenge/.conda/envs/wiki_dump/lib/python3.9/multiprocessing/pool.py", line 131, in worker
    put((job, i, result))
  File "/home/boulenge/.conda/envs/wiki_dump/lib/python3.9/multiprocessing/queues.py", line 377, in put
    self._writer.send_bytes(obj)
  File "/home/boulenge/.conda/envs/wiki_dump/lib/python3.9/multiprocessing/connection.py", line 200, in send_bytes
    self._send_bytes(m[offset:offset + 

**Preprocessing**

It included all prperocessing parts but the merging

In [10]:
for lang in langs:
    for side_context_size in side_context_sizes:
        print(f"Processing language: {lang} with side context size {side_context_size}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
        output_dir = f"{contexts_splitted_data_dir}/{lang}/context_size_{side_context_size}"
        
        subprocess.run([
            "python", "preprocessing.py",
            "--revisions_path", save_path,
            "--parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v2",
            "--first_versions_specific_parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v1",
            "--n_processes", "1",#str(processes),
            "--output_dir", output_dir,
            "--side_contexts_size", str(side_context_size),
            "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
            "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet"
        ])

        #preprocessed_data_path = os.path.join(output_dir, f'test_data.parquet')

        #preprocessed_data = pd.read_parquet(preprocessed_data_path)
        #display(preprocessed_data.head(5))

Processing language: fr with side context size 5
Running on 1 processes


100%|██████████| 24/24 [00:00<00:00, 4477.11it/s]


The length of the output is 24
Processing language: es with side context size 5
Running on 1 processes


100%|██████████| 21/21 [00:00<00:00, 2757.42it/s]


The length of the output is 21
Processing language: en with side context size 5
Running on 1 processes


100%|██████████| 18/18 [00:00<00:00, 6028.22it/s]


The length of the output is 18
Processing language: de with side context size 5
Running on 1 processes


100%|██████████| 670/670 [00:00<00:00, 6889.19it/s]


The length of the output is 670


**Merging**

In [11]:
test_data_dir = contexts_splitted_data_dir
output_dir = ready_to_score_data_dir

subprocess.run([
    "python", "merge_data.py",
    "--test_data_dir", test_data_dir,
    "--raw_data_dir", simple_render_raw_data_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--output_dir", output_dir,
    "--ignore_warnings"
])

first_lang = langs[0]
first_context_size = str(side_context_sizes[0])
scored_files_example = os.listdir(output_dir)
scored_files_example = list(filter(lambda x: (first_lang in x and first_context_size in x), scored_files_example))
final_input_data_path = os.path.join(output_dir, scored_files_example[0])

final_input_data_df = pd.read_parquet(final_input_data_path)
display(final_input_data_df.head(5))

Saving mention maps
Processing language fr (1/4)
Finished processing language fr
Processing language es (2/4)
Finished processing language es
Processing language en (3/4)
Finished processing language en
Processing language de (4/4)
Finished processing language de
Processing language fr (1/4)
Loaded pages
Loaded redirects


100%|██████████| 2515482/2515482 [00:08<00:00, 287297.63it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 24 links for language fr
Removed 0 links because of empty contexts
Removed 4 links because target title is not in source titles
Removed 5 links because of missing section
After cleaning, there are 15 links for language fr
Finished processing language fr
Processing language es (2/4)
Loaded pages
Loaded redirects


100%|██████████| 1829995/1829995 [00:07<00:00, 260146.48it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 21 links for language es
Removed 0 links because of empty contexts
Removed 5 links because target title is not in source titles
Removed 1 links because of missing section
After cleaning, there are 15 links for language es
Finished processing language es
Processing language en (3/4)
Loaded pages
Loaded redirects


100%|██████████| 6708565/6708565 [00:32<00:00, 206511.07it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 18 links for language en
Removed 0 links because of empty contexts
Removed 1 links because target title is not in source titles
Removed 0 links because of missing section
After cleaning, there are 17 links for language en
Finished processing language en
Processing language de (4/4)
Loaded pages
Loaded redirects


100%|██████████| 2826417/2826417 [00:10<00:00, 275864.01it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 670 links for language de
Removed 0 links because of empty contexts
Removed 156 links because target title is not in source titles
Removed 443 links because of missing section
After cleaning, there are 71 links for language de
Finished processing language de


,source_title,source_ID,target_title,context,insert_indices,moved_insertion_marker,link_sentence,link_text,section,mention_present,source_lead,first_version,second_version,direct_match,missing_category,all_contexts,target_lead
0,Artillerie,57485,Arbal%C3%A8te,None,[21],None,L’arbalète et la baliste sont apparues pendant...,arbalète,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,"L'arbalète est une arme de trait, constituée d..."
1,Artillerie,57485,Baliste,None,[21],None,L’arbalète et la baliste sont apparues pendant...,baliste,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,La baliste était un engin de siège développé à...
2,Artillerie,57485,Mausol%C3%A9e_de_l%27empereur_Qin,None,[24],None,Des exemplaires de ces «&nbsp;armes du diable&...,armée de terre cuite,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,Le mausolée de l'empereur Qin se trouve à prox...
3,Artillerie,57485,Qin_Shi_Huang,None,[24],None,Des exemplaires de ces «&nbsp;armes du diable&...,Qin shi Huang,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,"Qin Shi Huang, ou Qin Shi Huangdi, fut d'abord..."
4,Artillerie,57485,Chu_ko_nu,None,[25],None,"J.-C.), dont une version améliorée pour tirer ...",chu ko nu,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,La chu ko nu ou arbalète à répétition est une ...


**Scoring the contexts**

In [28]:
data_dir = ready_to_score_data_dir
output_dir = scored_data_dir

for lang in langs:
    for side_context_size in side_context_sizes:
        print(f"Processing language: {lang} with side context size {side_context_size}")

        subprocess.run([
            "python", "get_llm_scores.py",
            "--data_dir", data_dir,
            "--output_dir", output_dir,
            "--lang", lang,
            "--side_contexts_size", str(side_context_size),
            "--model_dir", "/dlabdata1/tsoares/models",
            "--model_prefix", "roberta_full",
            "--use_section_title",
            "--batch_size", "256",
            "--use_mentions",
            "--use_cuda",
            "--multilingual",
            "--multilingual_name", "multilingual-all"
        ])

scored_files_example = os.listdir(output_dir)[0]
scored_data = pd.read_parquet(f"{output_dir}/{scored_files_example}")

exemple = scored_data['all_contexts'][0][0]

for key in exemple.keys():
    print(f"{key}:\n{exemple[key]}\n")

Processing language: fr with side context size 5
Processing context size 5


100%|██████████| 17/17 [02:45<00:00,  9.73s/it]


Processing language: es with side context size 5
Processing context size 5


100%|██████████| 69/69 [11:11<00:00,  9.73s/it]


Processing language: en with side context size 5
Processing context size 5


100%|██████████| 3/3 [00:25<00:00,  8.50s/it]


Processing language: de with side context size 5
Processing context size 5


100%|██████████| 95/95 [15:25<00:00,  9.75s/it]


context:
On appelle artillerie l'ensemble des armes collectives ou lourdes — ou des troupes qui mettent en œuvre ces armes — servant à envoyer, à courte, moyenne ou grande distance, sur l'ennemi ou sur ses positions et ses équipements, divers projectiles de moyen ou gros calibre&nbsp;: obus, boulet, roquette, missile, pour appuyer ses propres troupes engagées dans une bataille terrestre, navale, antiaérienne ou un siège. Le terme serait apparu vers le XIIIe&nbsp;siècle, dérivant d’un mot du vieux français «&nbsp;artillier&nbsp;» qui servait à désigner les artisans fabricants d'armes et équipements de guerre. Ces artisans ont été pendant longtemps les seuls spécialistes dans le service de ces armes puisqu'ils les fabriquaient et les essayaient avant livraison. C'est pourquoi, jusqu'au XVIIIe&nbsp;siècle, ils étaient commissionnés par les souverains pour servir ces armes à la guerre. Canon exposé devant la basilique Notre-Dame de Montréal, Québec. Ainsi et par extension, le nom d'artille

**Colouring the pages**

In [36]:
output_dir = os.path.join(server_out_dir, "colored_pages")

# Run the coloring script
subprocess.run([
    "python", "color_pages.py",
    "--scored_data_dir", scored_data_dir,
    "--html_dir", crawling_output_dir,
    "--files_link_output_dir", output_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--extra_crawled_files_dir", crawling_output_dir,
    "--full_model_name", "roberta_full_multilingual-all"
])


Processing language fr with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1
Style: highlight


100%|██████████| 15/15 [00:02<00:00,  5.17it/s]


Processing language es with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1
Style: highlight


100%|██████████| 15/15 [00:06<00:00,  2.17it/s]


Processing language en with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1
Style: highlight


...

Processing language de with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1


  0%|          | 0/71 [00:00<?, ?it/s]

Style: highlight


100%|██████████| 71/71 [00:17<00:00,  4.03it/s]


CompletedProcess(args=['python', 'color_pages.py', '--scored_data_dir', '/dlabdata1/boulenge/wikidumps_data/report_examples_2/scored_data', '--html_dir', '/scratch/boulenge/report_examples_2/crawled_pages', '--files_link_output_dir', '/scratch/boulenge/report_examples_2/colored_pages', '--langs', 'fr,es,en,de', '--side_contexts_sizes', '5', '--extra_crawled_files_dir', '/scratch/boulenge/report_examples_2/crawled_pages', '--full_model_name', 'roberta_full_multilingual-all'], returncode=0)